In [ ]:
!pip install -q streamlit==1.28.0 pandas seaborn matplotlib
!npm install localtunnel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2;

In [ ]:
%%writefile app_Amer.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


st.set_page_config(page_title="Assignment 1: EDA & Data Cleaning", layout="wide")

st.sidebar.title("Assignment 1: EDA & Data Cleaning")

st.title("📊 Exploratory Data Analysis & Data Cleaning")
st.markdown("""
Welcome! This application helps you perform **Exploratory Data Analysis (EDA)**
and **Data Cleaning** interactively using Streamlit.

Please upload a CSV file to get started.
""")

uploaded_file = st.sidebar.file_uploader("Upload CSV File", type=["csv"])

if uploaded_file is not None:
    df = pd.read_csv(uploaded_file)
    df_cleaned = df.copy()

    tab_overview, tab_cleaning, tab_viz = st.tabs([
        "📋 Data Overview",
        "🧹 Data Cleaning",
        "📈 Visualizations"
    ])

    with tab_overview:
        st.header("Dataset Overview")

        col1, col2 = st.columns(2)
        with col1:
            st.subheader("Shape")
            st.write(f"**Rows:** {df.shape[0]}  |  **Columns:** {df.shape[1]}")
        with col2:
            st.subheader("Columns")
            st.write(", ".join(list(df.columns)))

        st.subheader("DataFrame Preview")
        st.dataframe(df.head(10), use_container_width=True)

        st.subheader("Descriptive Statistics")
        st.dataframe(df.describe(), use_container_width=True)

    with tab_cleaning:
        st.header("Missing Values Handling")

        missing_before = df.isnull().sum()
        total_missing_before = missing_before.sum()

        st.subheader("Before Cleaning")
        st.write(f"**Total Missing Values:** {total_missing_before}")
        if total_missing_before > 0:
            missing_df = missing_before[missing_before > 0].reset_index()
            missing_df.columns = ["Column", "Missing Count"]
            st.dataframe(missing_df, use_container_width=True)
        else:
            st.success("No missing values found in the dataset!")

        st.markdown("---")
        st.subheader("Cleaning Options")

        clean_option = st.radio(
            "Select a method to handle missing values:",
            options=["No Action", "Drop NA", "Fill with Mean (numeric only)", "Fill with Median (numeric only)"],
            index=0
        )

        if clean_option == "Drop NA":
            df_cleaned = df.dropna()
            st.info(f"Rows after dropping NA: {df_cleaned.shape[0]} (from {df.shape[0]})")
        elif clean_option == "Fill with Mean (numeric only)":
            df_cleaned = df.copy()
            numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns
            df_cleaned[numeric_cols] = df_cleaned[numeric_cols].fillna(df_cleaned[numeric_cols].mean())
            st.info("Missing values in numeric columns have been filled with the mean.")
        elif clean_option == "Fill with Median (numeric only)":
            df_cleaned = df.copy()
            numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns
            df_cleaned[numeric_cols] = df_cleaned[numeric_cols].fillna(df_cleaned[numeric_cols].median())
            st.info("Missing values in numeric columns have been filled with the median.")
        else:
            df_cleaned = df.copy()
            st.info("No cleaning action selected. Using original dataset.")

        st.markdown("---")
        st.subheader("After Cleaning")
        missing_after = df_cleaned.isnull().sum()
        total_missing_after = missing_after.sum()
        st.write(f"**Total Missing Values:** {total_missing_after}")
        if total_missing_after > 0:
            missing_after_df = missing_after[missing_after > 0].reset_index()
            missing_after_df.columns = ["Column", "Missing Count"]
            st.dataframe(missing_after_df, use_container_width=True)
        else:
            st.success("No missing values remain.")


    with tab_viz:
        st.header("EDA Visualizations")

        numeric_df = df_cleaned.select_dtypes(include=[np.number])

        st.subheader("Correlation Heatmap")
        if not numeric_df.empty and numeric_df.shape[1] >= 2:
            fig, ax = plt.subplots(figsize=(14, 10))
            sns.heatmap(
                numeric_df.corr(),
                annot=True,
                cmap="coolwarm",
                fmt=".2f",
                linewidths=0.5,
                ax=ax
            )
            ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
            ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
            st.pyplot(fig)
        else:
            st.warning("The dataset does not have enough numeric columns to display a correlation heatmap.")

        st.markdown("---")

        st.subheader("Histogram")
        if not numeric_df.empty:
            selected_col = st.selectbox("Select a numeric column for the histogram:", numeric_df.columns)
            fig2, ax2 = plt.subplots(figsize=(10, 5))
            sns.histplot(numeric_df[selected_col].dropna(), kde=True, color="skyblue", ax=ax2)
            ax2.set_title(f"Distribution of {selected_col}")
            st.pyplot(fig2)
        else:
            st.warning("No numeric columns are available for a histogram.")

        st.markdown("---")

        st.subheader("Categorical Bar Chart")
        categorical_df = df_cleaned.select_dtypes(include=["object", "category"])

        # Filter categorical columns with fewer than 20 unique values
        smart_cat_cols = [
            col for col in categorical_df.columns
            if categorical_df[col].nunique(dropna=True) < 20
        ]

        if smart_cat_cols:
            selected_cat_col = st.selectbox("Select a categorical column for the bar chart:", smart_cat_cols)
            value_counts = categorical_df[selected_cat_col].value_counts().head(20)
            fig3, ax3 = plt.subplots(figsize=(10, 5))
            sns.barplot(x=value_counts.index, y=value_counts.values, palette="viridis", ax=ax3)
            ax3.set_title(f"Category Distribution: {selected_cat_col}")
            ax3.set_xlabel(selected_cat_col)
            ax3.set_ylabel("Count")
            plt.xticks(rotation=45, ha="right")
            st.pyplot(fig3)
        else:
            st.info("No suitable categorical columns found. Categorical columns must have fewer than 20 unique values to be displayed here.")

else:
    st.info("👈 Please upload a CSV file via the sidebar to start the analysis.")


Overwriting app_Amer.py


In [ ]:
import urllib
print("Password/Enpoint IP for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))

!streamlit run app_Amer.py --server.enableCORS false --server.enableXsrfProtection false & npx localtunnel --port 8501

Password/Enpoint IP for localtunnel is: 34.23.28.115
⠙your url is: https://common-heads-flash.loca.lt



  You can now view your Streamlit app in your browser.

  Network URL: http://172.28.0.12:8501
  External URL: http://34.23.28.115:8501

/content/app_Amer.py:152: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=value_counts.index, y=value_counts.values, palette="viridis", ax=ax3)
/content/app_Amer.py:152: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=value_counts.index, y=value_counts.values, palette="viridis", ax=ax3)
/content/app_Amer.py:152: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and 